# YTSeg Transcript Pipeline Evaluation

This notebook evaluates the AI Service transcript pipelines on YTSeg directly, without a local Drive `audio/` folder. It selects 300 YTSeg samples from `retkowski/ytseg` `text/test`, streams matching rows from `audio/test`, runs 150 samples with `vad-chunked` and 150 samples with `diarized-turns`, then writes WER/CER/RTF metrics to Google Drive.

Outputs are resumable: rerunning the batch skips samples that already have metrics unless `FORCE_RERUN=True`.


In [ ]:
# Cell 1: clone repo and sync project dependencies with uv.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/vietanh-io/ai-video-content-management-system.git"
REPO_DIR = Path("/content/vid-pilot")
AI_SERVICE_DIR = REPO_DIR / "ai-service"

os.chdir("/content")

if not Path("/root/.local/bin/uv").exists():
    subprocess.run(
        "curl -LsSf https://astral.sh/uv/install.sh | sh",
        shell=True,
        check=True,
    )

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", "dev", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(AI_SERVICE_DIR)
subprocess.run(["uv", "sync"], check=True)

print("Repo ready:", REPO_DIR)


In [ ]:
# Cell 2: mount Drive and configure evaluation/runtime settings.
import os
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")

HF_TOKEN = userdata.get("HF_TOKEN") or userdata.get("HUGGING_FACE_HUB_TOKEN") or ""
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["PYANNOTE_AUTH_TOKEN"] = HF_TOKEN

OUTPUT_ROOT = Path("/content/drive/MyDrive/transcript_eval/ytseg_pipeline_300")
TMP_AUDIO_DIR = Path("/content/ytseg_tmp_audio")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

os.environ["TARGET_TOTAL"] = "300"
os.environ["VAD_COUNT"] = "150"
os.environ["DIARIZED_COUNT"] = "150"
os.environ["SEED"] = "42"
os.environ["FORCE_RERUN"] = "false"
os.environ["OUTPUT_ROOT"] = str(OUTPUT_ROOT)
os.environ["TMP_AUDIO_DIR"] = str(TMP_AUDIO_DIR)

os.environ["TRANSCRIPT_LANGUAGE"] = "en"
os.environ["ASR_PROVIDER"] = "faster-whisper"
os.environ["ASR_MODEL_SIZE"] = "large-v3-turbo"
os.environ["ASR_LANGUAGE"] = "en"
os.environ["ASR_DEVICE"] = "cuda"
os.environ["ASR_COMPUTE_TYPE"] = "float16"
os.environ["AUDIO_DECODER_PROVIDER"] = "torchaudio"
os.environ["VAD_PROVIDER"] = "silero"
os.environ["DIARIZATION_PROVIDER"] = "pyannote"
os.environ["DIARIZATION_DEVICE"] = "cuda"
os.environ["SOURCE_SEPARATION_PROVIDER"] = "noop"

print("Output root:", OUTPUT_ROOT)
print("HF token:", "ok" if HF_TOKEN else "missing")
print("ASR:", os.environ["ASR_MODEL_SIZE"], os.environ["ASR_DEVICE"], os.environ["ASR_COMPUTE_TYPE"])


In [ ]:
# Cell 3: write the YTSeg streaming evaluation runner.
from pathlib import Path

RUNNER_PATH = Path("/content/vid-pilot/ai-service/ytseg_transcript_pipeline_eval.py")

RUNNER_PATH.write_text(
r'''import csv
import json
import os
import random
import re
import statistics
import traceback
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import soundfile as sf
from datasets import Audio, load_dataset
from tqdm.auto import tqdm

from app.core.config import get_settings
from app.runtime.container import build_transcription_workflow
from app.schemas.transcription_request import TranscriptionOptionsInput, TranscriptionRequest

DATASET_NAME = "retkowski/ytseg"
TEXT_CONFIG = "text"
AUDIO_CONFIG = "audio"
DATASET_SPLIT = "test"

TARGET_TOTAL = int(os.environ.get("TARGET_TOTAL", "300"))
VAD_COUNT = int(os.environ.get("VAD_COUNT", "150"))
DIARIZED_COUNT = int(os.environ.get("DIARIZED_COUNT", "150"))
SEED = int(os.environ.get("SEED", "42"))
FORCE_RERUN = os.environ.get("FORCE_RERUN", "false").lower() == "true"
LANGUAGE = os.environ.get("TRANSCRIPT_LANGUAGE", os.environ.get("ASR_LANGUAGE", "en"))

DEV_RATIO = 0.70
MAX_PER_CHANNEL = 5
MIN_DURATION_SECONDS = 6 * 60
MAX_DURATION_SECONDS = 30 * 60
MIN_CHAPTERS = 4
MAX_CHAPTERS = 16

OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", "/content/drive/MyDrive/transcript_eval/ytseg_pipeline_300"))
TMP_AUDIO_DIR = Path(os.environ.get("TMP_AUDIO_DIR", "/content/ytseg_tmp_audio"))
REFERENCE_ROOT = OUTPUT_ROOT / "references"
EXPECTED_ROOT = OUTPUT_ROOT / "expected"
RESULT_ROOT = OUTPUT_ROOT / "results"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
EXPORT_ROOT = OUTPUT_ROOT / "exports"

MANIFEST_PATH = OUTPUT_ROOT / "manifest.csv"
METADATA_PATH = OUTPUT_ROOT / "selected_ytseg_metadata.json"
ERRORS_PATH = OUTPUT_ROOT / "errors.jsonl"
SUMMARY_PATH = EXPORT_ROOT / "summary.json"
PER_SAMPLE_CSV_PATH = EXPORT_ROOT / "per_sample_metrics.csv"
FAILED_CSV_PATH = EXPORT_ROOT / "failed_samples.csv"

for path in [
    OUTPUT_ROOT,
    TMP_AUDIO_DIR,
    REFERENCE_ROOT,
    EXPECTED_ROOT,
    RESULT_ROOT / "vad-chunked",
    RESULT_ROOT / "diarized-turns",
    METRICS_ROOT / "vad-chunked",
    METRICS_ROOT / "diarized-turns",
    EXPORT_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)


def atomic_write_json(path: Path, payload: dict | list) -> None:
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def append_error(video_id: str, pipeline: str, error: BaseException | str) -> None:
    payload = {
        "videoId": video_id,
        "pipeline": pipeline,
        "createdAt": datetime.now(timezone.utc).isoformat(),
        "errorType": type(error).__name__ if not isinstance(error, str) else "Error",
        "error": str(error),
        "traceback": traceback.format_exc(),
    }
    with ERRORS_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(payload, ensure_ascii=False) + "\n")


def parse_timestamp(value) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip()
    if not text:
        return 0.0
    if ":" not in text:
        return float(text)
    seconds = 0.0
    for part in [float(part) for part in text.split(":")]:
        seconds = seconds * 60 + part
    return seconds


def normalize_chapter_starts(raw_timestamps, duration_seconds: float) -> list[float]:
    starts = [parse_timestamp(value) for value in (raw_timestamps or [])]
    starts = sorted({round(start, 3) for start in starts if 0 <= start < duration_seconds})
    if not starts or starts[0] > 0.5:
        starts = [0.0, *starts]
    else:
        starts[0] = 0.0
    return starts


def duration_bucket(duration_seconds: float) -> str:
    if duration_seconds < 10 * 60:
        return "06_10"
    if duration_seconds < 15 * 60:
        return "10_15"
    if duration_seconds < 20 * 60:
        return "15_20"
    return "20_30"


def chapter_bucket(chapter_count: int) -> str:
    if chapter_count <= 6:
        return "04_06"
    if chapter_count <= 10:
        return "07_10"
    return "11_16"


def extract_reference_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = []
        for item in value:
            text = extract_reference_text(item)
            if text:
                parts.append(text)
        return " ".join(parts).strip()
    if isinstance(value, dict):
        for key in ["text_ref", "text_wt", "text_wl", "transcript", "transcripts", "full_text", "fullText", "text", "caption", "captions", "subtitle", "subtitles", "segments", "sentences"]:
            if key in value:
                text = extract_reference_text(value[key])
                if text:
                    return text
    return ""


def reference_text_from_row(row: dict) -> str:
    for key in ["text_ref", "text_wt", "text_wl", "transcript", "transcripts", "text", "captions", "subtitles", "segments", "sentences"]:
        if key in row:
            text = extract_reference_text(row[key])
            if text:
                return text
    return ""


def expected_payload(row: dict, pipeline: str) -> dict:
    video_id = row["video_id"]
    duration_seconds = float(row.get("duration") or 0.0)
    starts = normalize_chapter_starts(row.get("chapter_timestamps") or [], duration_seconds)
    titles = list(row.get("chapter_titles") or [])
    chapters = []
    for index, start_time in enumerate(starts):
        chapters.append({"index": index + 1, "startTime": round(float(start_time), 3), "title": titles[index] if index < len(titles) else None})
    return {
        "videoId": video_id,
        "source": "ytseg",
        "datasetSplit": DATASET_SPLIT,
        "evaluationPipeline": pipeline,
        "durationSeconds": round(duration_seconds, 3),
        "speakerCategory": row.get("speaker_category"),
        "channelId": row.get("channel_id"),
        "chapters": chapters,
    }


def candidate_from_row(row: dict) -> dict | None:
    duration_seconds = float(row.get("duration") or 0.0)
    chapter_timestamps = list(row.get("chapter_timestamps") or [])
    chapter_count = len(chapter_timestamps)
    channel_id = row.get("channel_id") or ""
    speaker_category = row.get("speaker_category") or "unknown"
    reference_text = reference_text_from_row(row)
    if not (MIN_DURATION_SECONDS <= duration_seconds <= MAX_DURATION_SECONDS):
        return None
    if not (MIN_CHAPTERS <= chapter_count <= MAX_CHAPTERS):
        return None
    if not channel_id:
        return None
    if not reference_text:
        return None
    return {
        "video_id": row["video_id"],
        "duration_seconds": duration_seconds,
        "chapter_count": chapter_count,
        "channel_id": channel_id,
        "speaker_category": speaker_category,
        "bucket": (duration_bucket(duration_seconds), chapter_bucket(chapter_count), speaker_category),
    }


def select_balanced_candidates(candidates: list[dict], target_total: int) -> list[dict]:
    rng = random.Random(SEED)
    shuffled = list(candidates)
    rng.shuffle(shuffled)
    by_bucket = defaultdict(list)
    for candidate in shuffled:
        by_bucket[candidate["bucket"]].append(candidate)
    selected = []
    channel_counts = Counter()
    bucket_order = sorted(by_bucket)
    while len(selected) < target_total:
        made_progress = False
        for bucket in bucket_order:
            items = by_bucket[bucket]
            while items:
                candidate = items.pop()
                channel_id = candidate["channel_id"]
                if channel_counts[channel_id] >= MAX_PER_CHANNEL:
                    continue
                selected.append(candidate)
                channel_counts[channel_id] += 1
                made_progress = True
                break
            if len(selected) >= target_total:
                break
        if not made_progress:
            break
    if len(selected) < target_total:
        raise ValueError(f"Only selected {len(selected)} videos; requested {target_total}. No transcript reference text was found. Expected YTSeg fields include text_ref, text_wt, or text_wl.")
    rng.shuffle(selected)
    return selected


def write_manifest(rows: list[dict]) -> None:
    if not rows:
        return
    with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def load_manifest() -> list[dict]:
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def build_or_load_manifest() -> list[dict]:
    if MANIFEST_PATH.exists() and not FORCE_RERUN:
        print("Using existing manifest:", MANIFEST_PATH)
        return load_manifest()
    print("Loading YTSeg text split for candidate selection...")
    text_ds = load_dataset(DATASET_NAME, TEXT_CONFIG, split=DATASET_SPLIT)
    text_rows_by_id = {row["video_id"]: row for row in text_ds}
    candidates = [candidate for row in text_rows_by_id.values() if (candidate := candidate_from_row(row))]
    selected = select_balanced_candidates(candidates, TARGET_TOTAL)
    pipeline_by_id = {}
    for candidate in selected[:VAD_COUNT]:
        pipeline_by_id[candidate["video_id"]] = "vad-chunked"
    for candidate in selected[VAD_COUNT:VAD_COUNT + DIARIZED_COUNT]:
        pipeline_by_id[candidate["video_id"]] = "diarized-turns"
    rows = []
    metadata_rows = []
    for candidate in selected[:VAD_COUNT + DIARIZED_COUNT]:
        video_id = candidate["video_id"]
        row = text_rows_by_id[video_id]
        pipeline = pipeline_by_id[video_id]
        reference_text = reference_text_from_row(row)
        expected = expected_payload(row, pipeline)
        reference_path = REFERENCE_ROOT / f"{video_id}.txt"
        expected_path = EXPECTED_ROOT / f"{video_id}.json"
        result_path = RESULT_ROOT / pipeline / f"{video_id}.json"
        metric_path = METRICS_ROOT / pipeline / f"{video_id}.json"
        reference_path.write_text(reference_text, encoding="utf-8")
        atomic_write_json(expected_path, expected)
        rows.append({
            "video_id": video_id,
            "pipeline": pipeline,
            "duration_seconds": expected["durationSeconds"],
            "chapter_count": len(expected["chapters"]),
            "speaker_category": row.get("speaker_category") or "",
            "channel_id": row.get("channel_id") or "",
            "reference_path": str(reference_path),
            "expected_path": str(expected_path),
            "result_path": str(result_path),
            "metric_path": str(metric_path),
            "status": "done" if metric_path.exists() else "pending",
        })
        metadata_row = dict(row)
        metadata_row["evaluation_pipeline"] = pipeline
        metadata_rows.append(metadata_row)
    write_manifest(rows)
    atomic_write_json(METADATA_PATH, {"createdAt": datetime.now(timezone.utc).isoformat(), "dataset": DATASET_NAME, "config": TEXT_CONFIG, "split": DATASET_SPLIT, "targetTotal": TARGET_TOTAL, "vadCount": VAD_COUNT, "diarizedCount": DIARIZED_COUNT, "rows": metadata_rows})
    print("Manifest written:", MANIFEST_PATH)
    return rows


def audio_input_path(video_id: str, audio_value: dict) -> Path:
    source_path = audio_value.get("path")
    if source_path and Path(source_path).exists():
        return Path(source_path)
    audio_bytes = audio_value.get("bytes")
    if audio_bytes:
        suffix = Path(source_path or "audio.wav").suffix or ".wav"
        temp_path = TMP_AUDIO_DIR / f"{video_id}{suffix}"
        temp_path.write_bytes(audio_bytes)
        return temp_path
    array = audio_value["array"]
    sampling_rate = int(audio_value["sampling_rate"])
    temp_path = TMP_AUDIO_DIR / f"{video_id}.wav"
    sf.write(temp_path, array, sampling_rate)
    return temp_path


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"[^0-9a-zA-Z\u00C0-\u1EF9\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def edit_distance(reference, hypothesis) -> int:
    previous = list(range(len(hypothesis) + 1))
    for i, ref_item in enumerate(reference, start=1):
        current = [i]
        for j, hyp_item in enumerate(hypothesis, start=1):
            current.append(min(current[j - 1] + 1, previous[j] + 1, previous[j - 1] + (ref_item != hyp_item)))
        previous = current
    return previous[-1]


def word_error_rate(reference_text: str, hypothesis_text: str) -> float:
    reference_words = normalize_text(reference_text).split()
    hypothesis_words = normalize_text(hypothesis_text).split()
    if not reference_words:
        return 0.0 if not hypothesis_words else 1.0
    return edit_distance(reference_words, hypothesis_words) / len(reference_words)


def character_error_rate(reference_text: str, hypothesis_text: str) -> float:
    reference = normalize_text(reference_text).replace(" ", "")
    hypothesis = normalize_text(hypothesis_text).replace(" ", "")
    if not reference:
        return 0.0 if not hypothesis else 1.0
    return edit_distance(list(reference), list(hypothesis)) / len(reference)


def segment_start(segment: dict) -> float:
    return float(segment.get("start_seconds") or segment.get("start") or 0.0)


def segment_end(segment: dict) -> float:
    return float(segment.get("end_seconds") or segment.get("end") or 0.0)


def average_segment_duration(segments: list[dict]) -> float | None:
    durations = [max(0.0, segment_end(segment) - segment_start(segment)) for segment in segments]
    return statistics.mean(durations) if durations else None


def run_transcript_pipeline(workflow, audio_path: Path, video_id: str, pipeline: str) -> tuple[dict, float]:
    import time
    started = time.perf_counter()
    result = workflow.execute(
        TranscriptionRequest(
            request_id=f"ytseg-{pipeline}-{video_id}",
            local_path=audio_path,
            filename=audio_path.name,
            content_type="audio/*",
            options=TranscriptionOptionsInput(language=LANGUAGE, enable_vad=True, enable_diarization=pipeline == "diarized-turns", enable_source_separation=False),
        )
    )
    return result.model_dump(), time.perf_counter() - started


def evaluate_one(workflow, manifest_row: dict, audio_path: Path) -> dict:
    video_id = manifest_row["video_id"]
    pipeline = manifest_row["pipeline"]
    result_path = Path(manifest_row["result_path"])
    metric_path = Path(manifest_row["metric_path"])
    reference_path = Path(manifest_row["reference_path"])
    duration_seconds = float(manifest_row["duration_seconds"])
    reference_text = reference_path.read_text(encoding="utf-8")
    payload, processing_time = run_transcript_pipeline(workflow, audio_path, video_id, pipeline)
    payload.update({"videoId": video_id, "source": "ytseg", "evaluationPipeline": pipeline, "durationSeconds": duration_seconds, "_eval": {"processingTimeSeconds": processing_time, "audioPath": str(audio_path)}})
    atomic_write_json(result_path, payload)
    hypothesis_text = payload.get("full_text") or " ".join(segment.get("text", "") for segment in payload.get("segments", []))
    segments = payload.get("segments", [])
    metric = {
        "video_id": video_id,
        "pipeline": pipeline,
        "status": "ok",
        "language": payload.get("language", ""),
        "asr_model": payload.get("asr_model", ""),
        "diarization_model": payload.get("diarization_model", ""),
        "duration_seconds": duration_seconds,
        "processing_time_seconds": processing_time,
        "rtf": processing_time / duration_seconds if duration_seconds > 0 else None,
        "segments_count": len(segments),
        "average_segment_duration_seconds": average_segment_duration(segments),
        "reference_words": len(normalize_text(reference_text).split()),
        "hypothesis_words": len(normalize_text(hypothesis_text).split()),
        "wer": word_error_rate(reference_text, hypothesis_text),
        "cer": character_error_rate(reference_text, hypothesis_text),
        "reference_path": str(reference_path),
        "result_path": str(result_path),
    }
    atomic_write_json(metric_path, metric)
    return metric


def write_failure_metric(manifest_row: dict, error: BaseException | str) -> dict:
    metric_path = Path(manifest_row["metric_path"])
    metric = {"video_id": manifest_row["video_id"], "pipeline": manifest_row["pipeline"], "status": "failed", "duration_seconds": float(manifest_row.get("duration_seconds") or 0.0), "reference_path": manifest_row.get("reference_path", ""), "result_path": manifest_row.get("result_path", ""), "error": str(error)}
    atomic_write_json(metric_path, metric)
    return metric


def print_metric(metric: dict) -> None:
    if metric.get("status") != "ok":
        print(metric["pipeline"], metric["video_id"], "FAILED", metric.get("error", "")[:300])
        return
    print(metric["pipeline"], metric["video_id"], "WER=", round(metric["wer"], 4), "CER=", round(metric["cer"], 4), "RTF=", round(metric["rtf"], 4) if metric.get("rtf") is not None else None, "segments=", metric["segments_count"])


def summarize_metrics(rows: list[dict], pipeline: str) -> dict:
    selected = [row for row in rows if row.get("pipeline") == pipeline]
    ok_rows = [row for row in selected if row.get("status") == "ok"]
    failed_rows = [row for row in selected if row.get("status") != "ok"]
    wers = [float(row["wer"]) for row in ok_rows]
    cers = [float(row["cer"]) for row in ok_rows]
    rtfs = [float(row["rtf"]) for row in ok_rows if row.get("rtf") is not None]
    def mean(values): return statistics.mean(values) if values else None
    def median(values): return statistics.median(values) if values else None
    def percentile(values, pct):
        if not values: return None
        ordered = sorted(values)
        return ordered[round((len(ordered) - 1) * pct)]
    return {"pipeline": pipeline, "attempted": len(selected), "success": len(ok_rows), "failed": len(failed_rows), "mean_wer": mean(wers), "median_wer": median(wers), "p90_wer": percentile(wers, 0.90), "mean_cer": mean(cers), "median_cer": median(cers), "p90_cer": percentile(cers, 0.90), "mean_rtf": mean(rtfs), "median_rtf": median(rtfs)}


def export_summary() -> None:
    metric_rows = [json.loads(path.read_text(encoding="utf-8")) for path in sorted(METRICS_ROOT.glob("*/*.json"))]
    summary = {"createdAt": datetime.now(timezone.utc).isoformat(), "dataset": DATASET_NAME, "config": AUDIO_CONFIG, "split": DATASET_SPLIT, "targetTotal": TARGET_TOTAL, "language": LANGUAGE, "pipelines": [summarize_metrics(metric_rows, "vad-chunked"), summarize_metrics(metric_rows, "diarized-turns")]}
    atomic_write_json(SUMMARY_PATH, summary)
    fieldnames = ["video_id", "pipeline", "status", "wer", "cer", "rtf", "duration_seconds", "processing_time_seconds", "segments_count", "average_segment_duration_seconds", "reference_words", "hypothesis_words", "asr_model", "diarization_model", "reference_path", "result_path", "error"]
    with PER_SAMPLE_CSV_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in metric_rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})
    with FAILED_CSV_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for row in metric_rows:
            if row.get("status") != "ok":
                writer.writerow({key: row.get(key, "") for key in fieldnames})
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("Summary:", SUMMARY_PATH)
    print("Per-sample CSV:", PER_SAMPLE_CSV_PATH)
    print("Failed CSV:", FAILED_CSV_PATH)


def main() -> None:
    manifest_rows = build_or_load_manifest()
    manifest_by_id = {row["video_id"]: row for row in manifest_rows}
    pending_ids = {row["video_id"] for row in manifest_rows if FORCE_RERUN or not Path(row["metric_path"]).exists() or row.get("status") != "done"}
    print("Manifest rows:", len(manifest_rows))
    print("Pending IDs:", len(pending_ids))
    if pending_ids:
        workflow = build_transcription_workflow(get_settings())
        audio_ds = load_dataset(DATASET_NAME, AUDIO_CONFIG, split=DATASET_SPLIT, streaming=True)
        audio_ds = audio_ds.cast_column("audio", Audio(decode=False))
        for row in tqdm(audio_ds, desc="Stream YTSeg audio rows"):
            video_id = row["video_id"]
            if video_id not in pending_ids:
                continue
            manifest_row = manifest_by_id[video_id]
            pipeline = manifest_row["pipeline"]
            metric_path = Path(manifest_row["metric_path"])
            if metric_path.exists() and not FORCE_RERUN:
                metric = json.loads(metric_path.read_text(encoding="utf-8"))
                if metric.get("status") == "ok":
                    manifest_row["status"] = "done"
                    pending_ids.discard(video_id)
                    write_manifest(manifest_rows)
                    if not pending_ids:
                        break
                    continue
            try:
                audio_path = audio_input_path(video_id, row["audio"])
                metric = evaluate_one(workflow, manifest_row, audio_path)
                manifest_row["status"] = "done"
                print_metric(metric)
            except Exception as error:
                manifest_row["status"] = "failed"
                append_error(video_id, pipeline, error)
                metric = write_failure_metric(manifest_row, error)
                print_metric(metric)
            finally:
                pending_ids.discard(video_id)
                write_manifest(manifest_rows)
                if not pending_ids:
                    break
    if pending_ids:
        raise ValueError(f"Missing IDs in streamed audio split: {sorted(pending_ids)}")
    export_summary()


if __name__ == "__main__":
    main()
''',
    encoding="utf-8",
)

print("Runner written:", RUNNER_PATH)


In [ ]:
# Cell 4: run the 300-sample evaluation.
# This uses uv for extra notebook-only dependencies. No pip.
import os
import subprocess
from pathlib import Path

RUNNER_PATH = Path("/content/vid-pilot/ai-service/ytseg_transcript_pipeline_eval.py")

cmd = [
    "uv",
    "run",
    "--with",
    "datasets",
    "--with",
    "soundfile",
    "--with",
    "tqdm",
    "--with",
    "huggingface_hub",
    "python",
    str(RUNNER_PATH),
]

completed = subprocess.run(
    cmd,
    cwd="/content/vid-pilot/ai-service",
    env=os.environ.copy(),
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"YTSeg transcript evaluation failed with exit code {completed.returncode}")


In [ ]:
# Cell 5: run VAD-chunked on the same 150 samples originally assigned to diarized-turns.
import csv
import os
import subprocess
from pathlib import Path

ORIGINAL_OUTPUT_ROOT = Path(os.environ["OUTPUT_ROOT"])
ORIGINAL_MANIFEST_PATH = ORIGINAL_OUTPUT_ROOT / "manifest.csv"
PAIRED_OUTPUT_ROOT = ORIGINAL_OUTPUT_ROOT / "paired_vad_on_diarized_samples"
PAIRED_MANIFEST_PATH = PAIRED_OUTPUT_ROOT / "manifest.csv"
RUNNER_PATH = Path("/content/vid-pilot/ai-service/ytseg_transcript_pipeline_eval.py")

if not ORIGINAL_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Original manifest not found: {ORIGINAL_MANIFEST_PATH}")

for path in [
    PAIRED_OUTPUT_ROOT,
    PAIRED_OUTPUT_ROOT / "results" / "vad-chunked",
    PAIRED_OUTPUT_ROOT / "metrics" / "vad-chunked",
    PAIRED_OUTPUT_ROOT / "exports",
]:
    path.mkdir(parents=True, exist_ok=True)

with ORIGINAL_MANIFEST_PATH.open("r", encoding="utf-8", newline="") as file:
    original_rows = list(csv.DictReader(file))

paired_rows = []
for row in original_rows:
    if row.get("pipeline") != "diarized-turns":
        continue
    video_id = row["video_id"]
    metric_path = PAIRED_OUTPUT_ROOT / "metrics" / "vad-chunked" / f"{video_id}.json"
    paired_row = dict(row)
    paired_row["pipeline"] = "vad-chunked"
    paired_row["result_path"] = str(PAIRED_OUTPUT_ROOT / "results" / "vad-chunked" / f"{video_id}.json")
    paired_row["metric_path"] = str(metric_path)
    paired_row["status"] = "done" if metric_path.exists() else "pending"
    paired_rows.append(paired_row)

if not paired_rows:
    raise ValueError("No diarized-turns rows found in the original manifest.")

with PAIRED_MANIFEST_PATH.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(paired_rows[0].keys()))
    writer.writeheader()
    writer.writerows(paired_rows)

paired_env = os.environ.copy()
paired_env["OUTPUT_ROOT"] = str(PAIRED_OUTPUT_ROOT)
paired_env["TARGET_TOTAL"] = str(len(paired_rows))
paired_env["VAD_COUNT"] = str(len(paired_rows))
paired_env["DIARIZED_COUNT"] = "0"
paired_env["FORCE_RERUN"] = "false"

cmd = [
    "uv",
    "run",
    "--with",
    "datasets",
    "--with",
    "soundfile",
    "--with",
    "tqdm",
    "--with",
    "huggingface_hub",
    "python",
    str(RUNNER_PATH),
]

print("Complement manifest:", PAIRED_MANIFEST_PATH)
print("Samples:", len(paired_rows))
print("Output root:", PAIRED_OUTPUT_ROOT)

completed = subprocess.run(
    cmd,
    cwd="/content/vid-pilot/ai-service",
    env=paired_env,
    text=True,
    capture_output=True,
)

print(completed.stdout)
if completed.stderr:
    print("STDERR:")
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"Paired VAD-on-diarized evaluation failed with exit code {completed.returncode}")


In [ ]:
# Cell 6: inspect summary and worst cases.
import csv
import json
import os
from pathlib import Path

OUTPUT_ROOT = Path(os.environ["OUTPUT_ROOT"])
SUMMARY_PATH = OUTPUT_ROOT / "exports" / "summary.json"
PER_SAMPLE_CSV_PATH = OUTPUT_ROOT / "exports" / "per_sample_metrics.csv"
FAILED_CSV_PATH = OUTPUT_ROOT / "exports" / "failed_samples.csv"

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

with PER_SAMPLE_CSV_PATH.open("r", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

ok_rows = [row for row in rows if row.get("status") == "ok"]
worst_wer = sorted(ok_rows, key=lambda row: float(row.get("wer") or 0.0), reverse=True)[:10]
worst_cer = sorted(ok_rows, key=lambda row: float(row.get("cer") or 0.0), reverse=True)[:10]

print("\nTop 10 WER")
for row in worst_wer:
    print(row["pipeline"], row["video_id"], "WER=", row["wer"], "CER=", row["cer"], "RTF=", row["rtf"])

print("\nTop 10 CER")
for row in worst_cer:
    print(row["pipeline"], row["video_id"], "WER=", row["wer"], "CER=", row["cer"], "RTF=", row["rtf"])

print("\nFiles:")
print("Summary:", SUMMARY_PATH)
print("Per sample:", PER_SAMPLE_CSV_PATH)
print("Failed:", FAILED_CSV_PATH)
